Anggota Kelompok:

1. Orchidea Savira Putri Arief Dhidin (25031554174)

2. Dentar Muhammad Ababilla (25031554277)

# **Exercise 5 Feature Engineering**
Try applying all the feature engineering on the text you have preprocessed from the previous exercise and analyze the given results.


## **1. Persiapan Environment & Import Library**

In [1]:
!pip install gensim umap-learn plotly -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 36.0 MB/s eta 0:00:00


In [2]:
import warnings
warnings.filterwarnings(action='ignore')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from tqdm import tqdm

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import gensim.models
from umap import UMAP

pd.set_option('display.max_columns', 12)
print("Semua library berhasil dimuat!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Semua library berhasil dimuat!


## **2. Load Hasil Preprocessing Exercise 4**


In [3]:
import requests, time, re, string
import pandas as pd
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt'); nltk.download('punkt_tab')
nltk.download('stopwords'); nltk.download('wordnet'); nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

quotes = []
for page in range(1, 6):
    r = requests.get(f"http://quotes.toscrape.com/page/{page}/")
    if r.status_code == 200:
        soup = BeautifulSoup(r.text, "html.parser")
        for box in soup.find_all("div", class_="quote"):
            quotes.append(box.find("span", class_="text").get_text(strip=True))
    time.sleep(0.3)
df_http = pd.DataFrame({"quote": quotes})
df_http["quote_final"] = df_http["quote"].apply(preprocess)
df_http.to_csv("hasil_preprocessing_http.csv", index=False)

records = []
for term in ["love", "dream", "night", "sky", "fire"]:
    try:
        res = requests.get("https://itunes.apple.com/search",
                           params={"term": term, "media": "music", "entity": "song", "limit": 3},
                           timeout=10)
        if res.status_code == 200:
            records.extend(res.json().get("results", []))
    except requests.exceptions.RequestException as e:
        print(term, "error:", e)
    time.sleep(0.3)
df_music = pd.DataFrame(records)
df_music["raw_text"] = df_music["trackName"].astype(str) + " " + df_music["artistName"].astype(str)
df_music["music_final"] = df_music["raw_text"].apply(preprocess)
df_music.to_csv("hasil_preprocessing_music_api.csv", index=False)

print(len(df_http), "quotes |", len(df_music), "lagu")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


50 quotes | 15 lagu


In [4]:
df_http = pd.read_csv('hasil_preprocessing_http.csv')
df_music = pd.read_csv('hasil_preprocessing_music_api.csv')

def finalize(series):
    s = series.fillna('').astype(str).apply(lambda t: re.sub(r'[^a-z\s]', ' ', t.lower()))
    return s.apply(lambda t: re.sub(r'\s+', ' ', t).strip())

df_http['prepro'] = finalize(df_http['quote_final'])
df_music['prepro'] = finalize(df_music['music_final'])

df_http = df_http[df_http['prepro'] != ''].reset_index(drop=True)
df_music = df_music[df_music['prepro'] != ''].reset_index(drop=True)

datasets = {'Quotes': df_http['prepro'], 'Music': df_music['prepro']}
for name, s in datasets.items():
    print(f"{name}: {len(s)} dokumen | rata-rata {s.str.split().str.len().mean():.1f} kata/dokumen")
    print("  contoh:", s.iloc[0])

Quotes: 50 dokumen | rata-rata 11.0 kata/dokumen
  contoh: world created process thinking changed without changing thinking
Music: 15 dokumen | rata-rata 3.1 kata/dokumen
  contoh: love keyshia cole


### **a. One Hot Encoding (Binary Bag of Words)**


In [5]:
ohe_results = {}
for name, s in datasets.items():
    vec = CountVectorizer(binary=True)
    X = vec.fit_transform(s)
    ohe_results[name] = (vec, X)
    df_ohe = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
    print(f"=== One Hot Encoding - {name} === shape: {df_ohe.shape}")
    display(df_ohe.head())

=== One Hot Encoding - Quotes === shape: (50, 321)


,ability,absolutely,act,actually,admit,adventure,...,worth,year,you,youer,youll,youre
0,0,0,0,0,0,0,...,0,0,0,0,0,0
1,1,0,0,0,0,0,...,0,0,0,0,0,0
2,0,0,0,0,0,0,...,0,0,0,0,0,0
3,0,0,0,0,0,0,...,0,0,0,0,0,0
4,0,1,0,0,0,0,...,0,0,0,0,0,0


=== One Hot Encoding - Music === shape: (15, 36)


,amsterdam,call,chance,cole,dream,einaudi,...,sister,sky,sleep,soulchild,table,zacari
0,0,0,0,1,0,0,...,0,0,0,0,0,0
1,0,0,0,0,0,0,...,0,0,0,0,0,1
2,0,0,0,0,0,0,...,0,0,0,1,0,0
3,0,0,0,0,1,0,...,0,0,0,0,0,0
4,0,0,0,0,1,0,...,0,0,0,0,0,0


### **b. Count Vectorizer (Bag of Words)**


In [6]:
bow_results = {}
for name, s in datasets.items():
    vec = CountVectorizer()
    X = vec.fit_transform(s)
    bow_results[name] = (vec, X)
    df_bow = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
    print(f"=== Count Vectorizer - {name} === shape: {df_bow.shape}")
    display(df_bow.head())

=== Count Vectorizer - Quotes === shape: (50, 321)


,ability,absolutely,act,actually,admit,adventure,...,worth,year,you,youer,youll,youre
0,0,0,0,0,0,0,...,0,0,0,0,0,0
1,1,0,0,0,0,0,...,0,0,0,0,0,0
2,0,0,0,0,0,0,...,0,0,0,0,0,0
3,0,0,0,0,0,0,...,0,0,0,0,0,0
4,0,2,0,0,0,0,...,0,0,0,0,0,0


=== Count Vectorizer - Music === shape: (15, 36)


,amsterdam,call,chance,cole,dream,einaudi,...,sister,sky,sleep,soulchild,table,zacari
0,0,0,0,1,0,0,...,0,0,0,0,0,0
1,0,0,0,0,0,0,...,0,0,0,0,0,1
2,0,0,0,0,0,0,...,0,0,0,1,0,0
3,0,0,0,0,1,0,...,0,0,0,0,0,0
4,0,0,0,0,1,0,...,0,0,0,0,0,0


### **c. TF-IDF**


In [7]:
tfidf_results = {}
for name, s in datasets.items():
    vec = TfidfVectorizer()
    X = vec.fit_transform(s)
    tfidf_results[name] = (vec, X)
    df_tfidf = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
    print(f"=== TF-IDF - {name} === shape: {df_tfidf.shape}")
    display(df_tfidf.head())

=== TF-IDF - Quotes === shape: (50, 321)


,ability,absolutely,act,actually,admit,adventure,...,worth,year,you,youer,youll,youre
0,0.00000,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0
1,0.41459,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0
2,0.00000,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0
3,0.00000,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0
4,0.00000,0.613264,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0


=== TF-IDF - Music === shape: (15, 36)


,amsterdam,call,chance,cole,dream,einaudi,...,sister,sky,sleep,soulchild,table,zacari
0,0.0,0.0,0.0,0.620116,0.000000,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000
1,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.466228
2,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.620116,0.0,0.000000
3,0.0,0.0,0.0,0.000000,0.612528,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000
4,0.0,0.0,0.0,0.000000,0.480535,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000


### **d. Analisis Hasil**


In [8]:
rows = []
for name in datasets:
    for method, store in [('One Hot', ohe_results), ('Count', bow_results), ('TF-IDF', tfidf_results)]:
        X = store[name][1]
        rows.append({'Dataset': name, 'Metode': method,
                     'Jumlah dokumen': X.shape[0], 'Jumlah fitur (vocab)': X.shape[1],
                     'Nilai non-zero': X.nnz,
                     'Sparsity (%)': round(100 * (1 - X.nnz / (X.shape[0] * X.shape[1])), 2)})
summary_bow = pd.DataFrame(rows)
display(summary_bow)

for name in datasets:
    vec, Xc = bow_results[name]
    freq = pd.Series(np.asarray(Xc.sum(axis=0)).ravel(), index=vec.get_feature_names_out()).sort_values(ascending=False)
    vec_t, Xt = tfidf_results[name]
    mean_tfidf = pd.Series(np.asarray(Xt.mean(axis=0)).ravel(), index=vec_t.get_feature_names_out()).sort_values(ascending=False)
    print(f"\n=== {name}: 10 kata paling sering (Count) vs 10 kata bobot rata-rata tertinggi (TF-IDF) ===")
    display(pd.DataFrame({'kata (Count)': freq.index[:10], 'frekuensi': freq.values[:10],
                          'kata (TF-IDF)': mean_tfidf.index[:10], 'rata-rata TF-IDF': mean_tfidf.values[:10].round(4)}))

    row = pd.Series(Xt[0].toarray().ravel(), index=vec_t.get_feature_names_out()).sort_values(ascending=False)
    print(f"Dokumen 0 ({name}): '{datasets[name].iloc[0]}'")
    print("Kata paling khas (TF-IDF):", row[row > 0].head(5).round(3).to_dict())

,Dataset,Metode,Jumlah dokumen,Jumlah fitur (vocab),Nilai non-zero,Sparsity (%)
0,Quotes,One Hot,50,321,455,97.17
1,Quotes,Count,50,321,455,97.17
2,Quotes,TF-IDF,50,321,455,97.17
3,Music,One Hot,15,36,47,91.30
4,Music,Count,15,36,47,91.30
5,Music,TF-IDF,15,36,47,91.30



=== Quotes: 10 kata paling sering (Count) vs 10 kata bobot rata-rata tertinggi (TF-IDF) ===


,kata (Count),frekuensi,kata (TF-IDF),rata-rata TF-IDF
0,love,15,love,0.0513
1,make,11,know,0.0398
2,life,10,life,0.0369
3,know,8,make,0.0349
4,dont,7,one,0.0312
5,never,7,read,0.0297
6,one,7,thing,0.0270
7,may,6,never,0.0269
8,go,6,great,0.0253
9,give,6,good,0.0248


Dokumen 0 (Quotes): 'world created process thinking changed without changing thinking'
Kata paling khas (TF-IDF): {'thinking': 0.613, 'changed': 0.339, 'changing': 0.339, 'created': 0.339, 'process': 0.339}

=== Music: 10 kata paling sering (Count) vs 10 kata bobot rata-rata tertinggi (TF-IDF) ===


,kata (Count),frekuensi,kata (TF-IDF),rata-rata TF-IDF
0,night,3,dream,0.1137
1,love,3,night,0.1021
2,fire,3,sky,0.0961
3,dream,3,fire,0.0913
4,sky,3,love,0.0882
5,pello,2,pello,0.0875
6,call,1,onerepublic,0.0527
7,amsterdam,1,lisa,0.0527
8,cole,1,sleep,0.0434
9,chance,1,cole,0.0413


Dokumen 0 (Music): 'love keyshia cole'
Kata paling khas (TF-IDF): {'cole': 0.62, 'keyshia': 0.62, 'love': 0.481}


**e. Tokenize Sentence**

In [9]:
sentences = {name: [word_tokenize(doc.lower()) for doc in tqdm(s)] for name, s in datasets.items()}
for name in sentences:
    vocab = set(w for doc in sentences[name] for w in doc)
    print(f"{name}: {len(sentences[name])} kalimat, {len(vocab)} kata unik, contoh: {sentences[name][0]}")

100%|██████████| 15/15 [00:00<00:00, 18797.30it/s]

Quotes: 50 kalimat, 323 kata unik, contoh: ['world', 'created', 'process', 'thinking', 'changed', 'without', 'changing', 'thinking']
Music: 15 kalimat, 36 kata unik, contoh: ['love', 'keyshia', 'cole']


### **f. Word Embedding (Word2Vec & FastText)**



In [10]:
from gensim.models import Word2Vec, FastText

tokenized_corpus = sentences['Quotes']

w2v_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=50,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=100,
    seed=42
)

ft_model = FastText(
    sentences=tokenized_corpus,
    vector_size=50,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=100,
    seed=42
)

print("Model Word2Vec & FastText berhasil dilatih!")
print("Jumlah kosakata dalam Word2Vec:", len(w2v_model.wv.key_to_index))
print("Jumlah kosakata dalam FastText:", len(ft_model.wv.key_to_index))

Model Word2Vec & FastText berhasil dilatih!
Jumlah kosakata dalam Word2Vec: 323
Jumlah kosakata dalam FastText: 323


### **g. Eksplorasi Word Embedding**


In [11]:
test_words = ['world', 'thinking', 'life']

for word in test_words:
    if word in w2v_model.wv:
        print(f"\nKata: '{word}'")
        print("  -> Mirip di Word2Vec:", [w for w, _ in w2v_model.wv.most_similar(word, topn=3)])
        print("  -> Mirip di FastText:", [w for w, _ in ft_model.wv.most_similar(word, topn=3)])
    else:
        print(f"\nKata '{word}' tidak ditemukan di kosakata.")


Kata: 'world'
  -> Mirip di Word2Vec: ['sister', 'come', 'theyre']
  -> Mirip di FastText: ['baby', 'anyway', 'anyone']

Kata: 'thinking'
  -> Mirip di Word2Vec: ['changed', 'changing', 'process']
  -> Mirip di FastText: ['making', 'changing', 'staring']

Kata: 'life'
  -> Mirip di Word2Vec: ['thousand', 'dy', 'reader']
  -> Mirip di FastText: ['balance', 'thousand', 'jojen']


### **h. Visualisasi Embedding menggunakan UMAP**


In [13]:
words = list(w2v_model.wv.key_to_index.keys())
vectors = np.array([w2v_model.wv[w] for w in words])

umap_reducer = UMAP(n_neighbors=5, min_dist=0.1, n_components=2, random_state=42)
coords = umap_reducer.fit_transform(vectors)

df_coords = pd.DataFrame({
    'word': words,
    'x': coords[:, 0],
    'y': coords[:, 1]
})

fig = px.scatter(df_coords, x='x', y='y', text='word', title='Visualisasi Word2Vec Word Embeddings')
fig.update_traces(textposition='top center')
fig.show()